# Fit Each File Separately

Multi-file workspace, **per-file independent fits**. This is the bridge between single-file basics (`01_basic_fitting`) and shared multi-file fitting (`21_project_level_shared_fit`).

**The setup is shared, the fits are not.** One model, one set of fit limits, one baseline window — applied across 6 files. Each file is fit independently; no parameters are tied across files.

**Why `Project` instead of a bare `for file in files: ...` loop?** Four concrete payoffs:

1. **Shared setup.** `project.set_fit_limits`, `project.define_baselines`, `project.load_models`, and `project.fit_baselines` apply once across all files — no per-file boilerplate to keep in sync.
2. **One coherent export tree.** `project.export_fits()` writes `<root>/<file_name>/<model>__<fit_type>/...` in a single call — easier to diff or zip than N separate per-file dumps.
3. **Cross-file comparison.** An unfiltered `project.results.compare_models()` surveys every file's slots in one DataFrame, ranked by metrics — useful for spotting outlier files at a glance.
4. **One portable HDF5.** `project.save_fits(path)` packages every file's slots into a single archive that round-trips via `FitResults.load`.

The data is the same kicked-decay dataset used by `21_project_level_shared_fit` (6 files with `expFun A` ranging from 5 down to 0.1) — loaded here via relative path so it isn't duplicated.

In [ ]:
import os
import numpy as np
import trspecfit

## 1. Load Six Files Into One Project

In [ ]:
project = trspecfit.Project(path=os.getcwd(), name="per_file_batch")

# Data lives next door so we don't duplicate the 6 CSVs in this example dir.
data_root = project.path.parent / "21_project_level_shared_fit" / "data"

energy = np.loadtxt(data_root / "energy.csv")
time = np.loadtxt(data_root / "time.csv")

shift_amplitudes = [5, 2, 1, 0.5, 0.2, 0.1]
files = []
for i, amp in enumerate(shift_amplitudes, start=1):
    file_data = np.loadtxt(data_root / f"data_{i}.csv", delimiter=",")
    f = trspecfit.File(
        parent_project=project,
        path=f"data_{i}",
        data=file_data,
        energy=energy,
        time=time,
    )
    files.append(f)
    print(f"File {i}: expFun A = {amp}, shape {file_data.shape}")

project.describe(detail=1)

## 2. Shared Setup Across All Files

Same fit limits, same baseline window, same model loaded onto every file — one call each. This is the per-file boilerplate that a bare `for f in files: ...` loop would have to repeat.

In [ ]:
project.set_fit_limits(energy_limits=[5, 18], time_limits=[-10, 99])
project.define_baselines(time_start=0, time_stop=12, time_type="ind")
project.load_models(model_yaml="models_energy.yaml", model_info="base")

## 3. Per-File Baseline Fits

`project.fit_baselines` is a thin loop over `file.fit_baseline` — each baseline is fit independently. It is **not** a shared fit; only the setup is shared.

In [ ]:
project.fit_baselines(model_name="base", stages=2)

## 4. Per-File 2D Fits (Independent Loop)

For the 2D step we deliberately use a per-file loop (`for f in files: f.fit_2d(...)`) instead of `project.fit_2d(...)`. The latter is the **shared-parameter** path (notebook `21_project_level_shared_fit`); here we want every file's `tau`, `SD`, `A` fit independently.

The 2D YAML uses `vary: project/file/static` strings (carried over from notebook 21). For per-file fits those are reduced to a simple True/False vary level (`project` and `file` both → True; `static` → False), so the same YAML works in either workflow.

In [ ]:
project.load_models(model_yaml="models_energy.yaml", model_info="2D")
project.add_time_dependences(
    target_model="2D",
    target_parameter="GLP_01_x0",
    dynamics_yaml="models_time.yaml",
    dynamics_model="MonoExpPosIRF",
)

for f in files:
    f.fit_2d(model_name="2D", stages=2)

## 5. Cross-File Comparison

Omit `file=` to survey the whole batch: `project.results.compare_models(fit_type="2d")` reads `_fit_history` (no disk I/O) and returns one row per `(file, model, fit_type)` slot. The `file=` filter is single-target — pass one file name or `File` to zoom into one row; iterate to inspect a subset.

With one model and one fit type per file, the comparison effectively becomes a **per-file fit-quality survey**: which files fit best, which have the largest residuals, which dynamics parameters drift outliers. For a true model-vs-model comparison see `10_model_comparison`.

In [ ]:
project.results.compare_models(fit_type="2d")

## 6. Export and Archive the Whole Batch

Two methods, two audiences:

- **`project.export_fits()` → CSV + PNG tree.** One coherent directory rooted at `<path>/<file_name>/<model>__<fit_type>/...` — easier to diff or zip than N per-file dumps. One-way; no `load` counterpart.
- **`project.save_fits()` → single HDF5 archive.** Lossless, σ-snapshot included, round-trips back into trspecfit via `FitResults.load`.

Both calls honor the same `(file, model, fit_type)` filters as `compare_models` — pass them when you want to ship a subset instead of the whole batch. See [`11_save_load_export`](../11_save_load_export/example.ipynb) for the filtered-export and round-trip details.

In [ ]:
project.export_fits("batch_export", overwrite=True)
project.save_fits("batch.fit.h5", overwrite=True)

loaded = trspecfit.FitResults.load("batch.fit.h5")
print(loaded)
print("\nfiles:  ", loaded.files())
print("models: ", loaded.models())

## Tips

- **Choose the right `Project` method.** `project.fit_baselines()` and `project.load_models()` are setup helpers — loops that share boilerplate. `project.fit_2d()` is a true shared-parameter fit (see `21_project_level_shared_fit`). Use a bare `for f in files: f.fit_2d(...)` loop for independent 2D fits, as shown here.
- **Survey vs zoom.** `compare_models()` with no `file=` returns one row per slot across the whole project — the cross-file survey. `compare_models(file=name)` filters to one file (single-target only; loop to inspect a subset).
- **Next step:** if any subset of parameters *should* be shared across files (instrument response, decay constants), graduate to `21_project_level_shared_fit` — the YAMLs already use the `vary: project/file/static` flags that drive `project.fit_2d`.